# BOT-2 Shape Scoring · Five-Fold Child-Grouped Cross-Validation

## 1. Setup

In [ ]:
%pip install -q -U timm "transformers>=4.56" safetensors

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, importlib
import pandas as pd

DRIVE_DIR = "/content/drive/MyDrive/Evaluation by prof. nam's research group"
FINAL_DIR = os.path.join(DRIVE_DIR, 'final')
ZIP_PATH = os.path.join(DRIVE_DIR, 'Shapes.zip')
SAVE_WEIGHTS = 'ensemble'         # weight files kept on Drive: 'ensemble' (the three DPE networks), 'all' or 'none'
SHUTDOWN_AT_END = True            # release the Colab runtime after the last cell and after any error

sys.path.insert(0, FINAL_DIR)
import bot2score
from bot2score import (utils, data, metrics, scorers, models, features, rules, classical, training, evaluation,
                       figures, verify)
for module in (utils, data, metrics, scorers, models, features, rules, classical, training, evaluation, figures, verify):
    importlib.reload(module)
utils.quiet_libraries()
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_colwidth', 120, 'display.precision', 3)

P = utils.make_paths(os.path.join(FINAL_DIR, 'results'))
log = utils.Log(os.path.join(P.logs, 'run_log.txt'))
if SHUTDOWN_AT_END:
    utils.release_runtime_on_error(log)
env = utils.environment()
utils.save_json(os.path.join(P.logs, 'environment.json'),
                {**env, 'bot2score': bot2score.__version__,
                 'code_sha256': utils.code_hashes(os.path.dirname(bot2score.__file__))})
log(f"bot2score {bot2score.__version__} | {env.get('gpu', 'no GPU')} | torch {env['torch']} | "
    f"timm {env['timm']} | transformers {env['transformers']}")

## 2. Data

In [ ]:
D = data.prepare(ZIP_PATH, P, log=log)
data.score_table(D, P)

In [ ]:
data.criterion_table(D, P)

In [ ]:
data.fold_table(D, P)

In [ ]:
D.dropped

## 3. Scorers

In [ ]:
models.scorer_table()

In [ ]:
models.network_table()

In [ ]:
models.recipe()

## 4. OpenCV geometric features

In [ ]:
G = features.compute(D, P, log=log)
features.table(G, P)

## 5. OpenCV rubric rules

In [ ]:
rules.run(D, G, P, log=log)

## 6. Machine-learning classifiers

In [ ]:
classical.run(D, G, P, log=log)

## 7. Deep networks

In [ ]:
training.run(D, P, save=SAVE_WEIGHTS, log=log)

## 8. Out-of-fold test predictions

In [ ]:
R = evaluation.collect(D, P, log=log)
evaluation.selection(P)

## 9. Test accuracy

In [ ]:
S = evaluation.summary(D, R, P)
S[['Scorer', 'Paradigm', 'Accuracy (%)', 'CI low', 'CI high', 'Fold SD', 'Accuracy, all drawings (%)',
   'Balanced accuracy', 'Macro F1', 'MAE']].round(3)

In [ ]:
T = evaluation.item_accuracy(D, R, P)
T.round(2)

In [ ]:
F = evaluation.fold_accuracy(D, R, P)
F.round(2)

In [ ]:
evaluation.paradigms(D, R, P).round(3)

In [ ]:
evaluation.ensemble(D, R, P).round(3)

## 10. Agreement with the expert scores

In [ ]:
A = evaluation.agreement(D, R, P)
A.round(3)

In [ ]:
IA = evaluation.item_agreement(D, R, P)
IA[IA.Key == scorers.PROPOSED].drop(columns=['Key', 'Scorer']).set_index('Item').round(3)

In [ ]:
M = evaluation.item_metrics(D, R, P)
evaluation.confusion(D, R, P)
M[M.Scorer == scorers.PROPOSED_LABEL].set_index('Item').round(3)

In [ ]:
RA = rules.criterion_agreement(D, P)
RA

## 11. Paired comparisons

In [ ]:
C = evaluation.compare(D, R, P)
C.drop(columns=['Key A', 'Key B'])

## 12. Drawings below full marks

In [ ]:
curves = evaluation.below_full_curves(D, R, P)
S[['Scorer', 'Precision below full', 'Recall below full', 'F1 below full', 'AP below full']].round(3)

## 13. Embeddings

In [ ]:
E = evaluation.embeddings(D, R, P, log=log)
E['separation']

## 14. Figures

In [ ]:
figures.setup()

In [ ]:
figures.scores(D, P, log=log);

In [ ]:
figures.criteria(data.criterion_table(D), RA, P, log=log);

In [ ]:
figures.accuracy(S, F, P, log=log);

In [ ]:
figures.agreement(IA, P, log=log);

In [ ]:
figures.comparisons(C, P, log=log);

In [ ]:
figures.paradigms(C, P, log=log);

In [ ]:
figures.item_accuracy(T, S, P, log=log);

In [ ]:
figures.precision_recall(D, R, S, curves, P, log=log);

In [ ]:
figures.confusion(D, R, P=P, log=log);

In [ ]:
figures.embeddings(E, P, log=log);

In [ ]:
figures.training(D, P, log=log);

## 15. Verification of the saved weights

In [ ]:
verify.check_weights(D, P, log=log)

In [ ]:
verify.manifest(P, log=log).tail()

## 16. Release the runtime

In [ ]:
log('Finished. All results are in', P.root)
if SHUTDOWN_AT_END:
    utils.release_runtime(log)